# Severity Classification with a Free, Unlimited LLM (local Ollama)

Severity (Low / Medium / High / Critical) is predicted at **predict time** by an LLM - it is a
completely separate model from the trained **root-cause classifier** (`rootcause_training.ipynb`).
The LLM receives the predicted ROOT CAUSE + a block of evidence log lines.

**Primary provider: local [Ollama](https://ollama.com)** - free, unlimited, no key, runs on your own machine.
**Optional:** Google Gemini free tier (`GEMINI_API_KEY`, rate-limited) used only when no local Ollama is up.
**Fallback:** deterministic policy (`ml/severity_policy.json`, root-cause baselines) when no LLM is available.

This notebook mirrors `ml/severity_llm.py`, which `predict.py` calls at runtime.

In [ ]:
# 1. No pip install needed - stdlib only.
print("stdlib only, no SDK required")
# Install the local model once:  ollama pull qwen2.5:3b   (then: ollama serve)

In [ ]:
# 2. Config
import getpass
import os

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:3b")
PROVIDER = os.environ.get("SEVERITY_LLM_PROVIDER", "ollama")  # ollama | gemini | none | auto
SEVERITY_LEVELS = ("Low", "Medium", "High", "Critical")

if not os.environ.get("GEMINI_API_KEY") and input("Set a Gemini API key as backup? (y/n): ").strip().lower().startswith("y"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Free key from https://aistudio.google.com/apikey: ")
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")

print(f"provider: {PROVIDER} | ollama: {OLLAMA_HOST} ({OLLAMA_MODEL}) | gemini key: {bool(os.environ.get('GEMINI_API_KEY'))}")

In [ ]:
# 3. Classifier (same logic as ml/severity_llm.py, inlined)
import json
import urllib.error
import urllib.request

GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models"

SYSTEM_PROMPT = (
    "You score the operational severity of a CI/CD build job that failed. "
    "You receive the predicted ROOT CAUSE plus a block of evidence log lines. "
    "Return ONLY a JSON object with two keys: "
    '{"severity": "Low"|"Medium"|"High"|"Critical", "reason": "<one short sentence>"}\n'
    "Rubric:\n"
    '- Low: cosmetic/warning-level issue, build continues; e.g. lint style notes, deprecation, flaky retry.\n'
    '- Medium: partial failure or transient problem that blocks a step but not the whole system; '
    "e.g. test flake, timeout with retry, warning that needs attention.\n"
    '- High: a real failure requiring a fix; e.g. compile/test/build failure, dependency install error, '
    "deployment error, auth/permission failure.\n"
    '- Critical: outage, security breach, data loss, production impact, leaked secret, disk full, '
    "fatal/crash of the whole pipeline or service.\n"
    "When in doubt prefer the boundary that the log text itself shows."
)


def user_text(root_cause, evidence):
    return f"root cause: {root_cause}\n\nevidence log lines:\n{evidence}"


def call_ollama(root_cause, evidence, model=OLLAMA_MODEL):
    body = {
        "model": model, "stream": False, "format": "json",
        "options": {"temperature": 0},
        "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": user_text(root_cause, evidence)}],
    }
    req = urllib.request.Request(
        OLLAMA_HOST + "/api/chat", data=json.dumps(body).encode("utf-8"),
        headers={"content-type": "application/json"}, method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read().decode("utf-8"))["message"]["content"]


def call_gemini(root_cause, evidence, model=GEMINI_MODEL):
    url = f"{GEMINI_URL}/{model}:generateContent?key={os.environ['GEMINI_API_KEY']}"
    body = {
        "system_instruction": {"parts": [{"text": SYSTEM_PROMPT}]},
        "contents": [{"parts": [{"text": user_text(root_cause, evidence)}]}],
        "generationConfig": {"temperature": 0, "maxOutputTokens": 200,
                             "responseMimeType": "application/json"},
    }
    req = urllib.request.Request(
        url, data=json.dumps(body).encode("utf-8"),
        headers={"content-type": "application/json"}, method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        if e.code != 400:
            raise
        del body["generationConfig"]["responseMimeType"]
        req = urllib.request.Request(
            url, data=json.dumps(body).encode("utf-8"),
            headers={"content-type": "application/json"}, method="POST",
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))
    return "".join(p.get("text", "") for p in data["candidates"][0]["content"]["parts"])


def ollama_up():
    try:
        with urllib.request.urlopen(OLLAMA_HOST + "/api/tags", timeout=1.5):
            return True
    except Exception:
        return False


def classify_severity_llm(root_cause, evidence):
    provider = PROVIDER
    if provider == "none":
        raise RuntimeError("SEVERITY_LLM_PROVIDER=none (disabled)")
    if provider == "ollama":
        raw, name = call_ollama(root_cause, evidence), "ollama"
    elif provider == "gemini":
        raw, name = call_gemini(root_cause, evidence), "gemini"
    else:
        if ollama_up():
            raw, name = call_ollama(root_cause, evidence), "ollama"
        elif os.environ.get("GEMINI_API_KEY"):
            raw, name = call_gemini(root_cause, evidence), "gemini"
        else:
            raise RuntimeError("no LLM available (no Ollama, no GEMINI_API_KEY)")
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("no JSON in LLM reply")
    obj = json.loads(raw[start:end + 1])
    sev = obj.get("severity")
    if sev not in SEVERITY_LEVELS:
        raise ValueError("bad severity: " + repr(sev))
    return sev, obj.get("reason", ""), name, raw


print("classifier defined")

## Deterministic fallback
If no Ollama, no Gemini key, a call error, or an unparseable reply: **tool code > upgrade/downgrade
signal > root-cause baseline** (mirrors `ml/severity_policy.json`).

In [ ]:
# 3b. Deterministic fallback policy (mirrors ml/severity_policy.json)
import re

ROOT_CAUSE_BASELINES = {
    "api_gateway_deployment_error": "High",
    "container_registry_server_error": "High",
    "dependency_installation_failure": "High",
    "helm_resource_error": "High",
    "misconfigured_env_variable": "High",
    "external_file_invalid_format": "Medium",
    "flaky_ui_test": "Medium",
    "git_transient_error": "Medium",
    "host_resolution_failure": "Medium",
    "job_execution_timeout": "Medium",
    "remote_call_timeout": "Medium",
    "runner_image_pull_failure": "Medium",
    "runner_pod_waiting_timeout": "Medium",
}
PYLINT_MAP = {"F": "Critical", "E": "High", "W": "Medium", "C": "Low", "R": "Medium"}
FLAKE8_MAP = {"E": "Medium", "W": "Low", "F": "Medium", "C": "Low"}
DOWNGRADE = ("flaky", "flakiness", "flaked", "retry", "retries", "retrying", "timeout",
             "timed out", "transient", "temporary")
UPGRADE = ("fatal", "permission denied", "access denied", "not authorized", "forbidden",
           "tls", "certificate", "expired", "cannot start container", "disk full")
RANK = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}
BANDIT = re.compile(r"Severity:\s*(Low|Medium|High|Undefined)", re.IGNORECASE)
PREFIX = re.compile(r"^\s*(?:[\w./\\-]+):\d+(?::\d+)?:\s*([A-Z])\d{3,4}(?::|\s)", re.IGNORECASE)
PYLINT_SUFFIX = re.compile(r"\([a-z0-9-]+\)\s*$", re.IGNORECASE)


def tool_severity(text):
    m = BANDIT.search(text)
    if m:
        return "Medium" if m.group(1).capitalize() == "Undefined" else m.group(1).capitalize()
    if text.lstrip().startswith(">> Issue:"):
        return "Low"
    m = PREFIX.search(text)
    if m:
        code = m.group(1).upper()
        table = PYLINT_MAP if PYLINT_SUFFIX.search(text) else FLAKE8_MAP
        return table.get(code)
    return None


def policy_severity(root_cause, text):
    t = tool_severity(text)
    if t:
        return t
    lower = text.lower()
    baseline = ROOT_CAUSE_BASELINES.get(root_cause, "Medium")
    if any(s in lower for s in UPGRADE):
        return "Critical"
    if any(s in lower for s in DOWNGRADE):
        return "Medium" if RANK[baseline] > RANK["Medium"] else baseline
    return baseline


def classify_line(root_cause, evidence, use_llm=True):
    if use_llm:
        try:
            sev, reason, name, raw = classify_severity_llm(root_cause, evidence)
            return {"severity": sev, "source": name, "reason": reason, "raw": raw}
        except Exception as e:
            print("[fallback]", type(e).__name__, str(e)[:120])
    return {"severity": policy_severity(root_cause, evidence), "source": "policy", "reason": "", "raw": ""}


print("fallback policy defined")

In [ ]:
# 4. Demo: (root cause, evidence block) - LLM verdict + policy for comparison
DEMOS = [
    ("misconfigured_env_variable",
     "ERROR: environment variable DATABASE_URL is not configured\nmake: *** [test] Error 1\nTraceback (most recent call last)"),
    ("git_transient_error",
     "fatal: unable to access 'https://gitlab.tinaa.osc.tac.net/t/repo.git/': The requested URL returned error: 500 (retried)"),
    ("host_resolution_failure",
     "ssh: Could not resolve hostname db.internal: Name or service not known\nERROR: PING db.internal failed - host unreachable"),
    ("job_execution_timeout",
     "timeout waiting for response after 30s (request #45123), retrying with backoff\nBuild timed out"),
    ("dependency_installation_failure",
     "ModuleNotFoundError: No module named 'psycopg2'\npip install psycopg2 failed with exit code 1"),
    ("flaky_ui_test",
     "FAILED tests/ui_test.py (flaky, passed on rerun)\nElement not clickable at point (400, 210)"),
]

for rc, ev in DEMOS:
    r = classify_line(rc, ev, use_llm=True)
    p = policy_severity(rc, ev)
    print(f"- [{r['severity']} | {r['source']}]   [{p} | policy]   {rc}")
    if r["reason"]:
        print(f"    reason: {r['reason'].strip()}")

In [ ]:
# 5. Agreement check on an expected-severity gold set
GOLD = [
    ("misconfigured_env_variable", "ERROR: environment variable DATABASE_URL is not configured", "High"),
    ("dependency_installation_failure", "ModuleNotFoundError: No module named 'psycopg2'", "High"),
    ("job_execution_timeout", "timeout waiting for response after 60s (request #8), retrying", "Medium"),
    ("runner_image_pull_failure", "Failed to pull image: manifest for image:x not found (retried)", "Medium"),
    ("host_resolution_failure", "Could not resolve host proxy.internal: Name or service not known", "Medium"),
    ("api_gateway_deployment_error", "ERROR: failed to deploy gateway config to cluster, aborting deploy", "High"),
    ("git_transient_error", "Shallow clone failed, retrying fetch from origin", "Medium"),
    ("container_registry_server_error", "ERROR: registry returned 500 for tag latest, pushing failed", "High"),
]

agree = 0
for rc, ev, exp in GOLD:
    r = classify_line(rc, ev, use_llm=True)
    ok = r["severity"] == exp
    agree += ok
    print(f"  {'OK ' if ok else 'ERR'}  llm={r['severity']:<8} exp={exp:<8} {rc}")
print(f"\nLLM agreement with expectations: {agree}/{len(GOLD)}")

## Wiring into the pipeline
- `predict.py` runs the root-cause model on the whole job log, then calls
  `ml.severity_llm.classify_severity(root_cause, evidence_block)` for the job's severity.
- **Free + unlimited:** install Ollama on the Jenkins machine, `ollama pull qwen2.5:3b`,
  `ollama serve` (works in WSL2 too). The model loads on first use, unloads ~5 min after idle.
- **Cloud backup:** free `GEMINI_API_KEY` (aistudio.google.com/apikey); used only when Ollama is down.
- **No LLM:** `SEVERITY_LLM_PROVIDER=none` -> root-cause-baseline policy, zero extra RAM.